<div style="background:linear-gradient(135deg,#143840 0%,#2B6264 100%);border-radius:14px;padding:32px 36px;color:#fff;font-family:'DM Sans',Arial,sans-serif;">
  <div style="font-size:11px;letter-spacing:2px;text-transform:uppercase;color:#FF4B31;font-weight:700;margin-bottom:10px;">Solutions Onboarding &middot; IBOR Training</div>
  <div style="font-size:30px;font-weight:700;line-height:1.15;margin-bottom:10px;">IBOR NB07 &mdash; Reconciliation</div>
  <div style="font-size:15px;color:rgba(255,255,255,.82);max-width:640px;line-height:1.55;">Reconcile IBOR positions against custodian-reported positions using the Reconciliation API, surfacing the deliberate quantity and missing-instrument breaks.</div>
</div>

<sub>IBOR pack sequence: NB00 &nbsp;&rarr;&nbsp; NB01 &nbsp;&rarr;&nbsp; NB02 &nbsp;&rarr;&nbsp; NB03 &nbsp;&rarr;&nbsp; NB04 &nbsp;&rarr;&nbsp; NB05 &nbsp;&rarr;&nbsp; NB06 &nbsp;&rarr;&nbsp; **NB07** &nbsp;&rarr;&nbsp; NB08</sub>

# NB07: Reconciliation

Reconciles IBOR positions against custodian reported positions using LUSID's Reconciliation API and Group Reconciliation UI.

The custodian data has **two deliberate breaks**:
1. **Amortising MBS**: Custodian reports 4,950,000 face vs IBOR 5,000,000 (missed principal paydown of 50,000)
2. **Convertible 2.5% March 2027**: Missing entirely from custodian report

In [ ]:
import subprocess, sys
subprocess.check_call([sys.executable, '-m', 'pip', 'install', 'lusid-sdk', 'lumipy', '-q'],
                      stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
import os, json
from datetime import datetime, timedelta, timezone
import pandas as pd
import lusid as lu
import lusid.models as lm
import lumipy as lp
pd.set_option("display.max_columns", None)
SCOPE = 'ibor-training-v9'

with open("secrets.json") as f: secrets = json.load(f)
pat = secrets.get("api", {}).get("accessToken")
lusid_factory = lu.extensions.SyncApiClientFactory(
    config_loaders=[lu.extensions.ArgsConfigurationLoader(
        api_url=secrets["api"]["lusidUrl"], access_token=pat)])
lumi = lp.get_client(access_token=pat, api_url=secrets["api"]["lusidUrl"].replace("/api", "/honeycomb"))

tp_api = lusid_factory.build(lu.TransactionPortfoliosApi)
portfolios_api = lusid_factory.build(lu.PortfoliosApi)
recon_api = lusid_factory.build(lu.ReconciliationsApi)
grp_recon_api = lusid_factory.build(lu.GroupReconciliationsApi)
print('Ready')

---
## Step 1: Create Flattened IBOR Portfolio
IBOR-FI has Sub Holding Keys that split positions by Strategy and CustodianAccount. The custodian reports flat positions. We create a flattened copy for comparison.

In [ ]:
# Get IBOR-FI actual holdings and aggregate across SHKs
fi_resp = tp_api.get_holdings(scope=SCOPE, code="IBOR-FI", effective_at="2024-09-30T00:00:00+00:00")

agg = {}
for h in fi_resp.values:
    if h.holding_type != 'P':
        continue
    luid = h.instrument_uid
    if luid not in agg:
        agg[luid] = {"units": 0, "cost": 0, "currency": h.currency}
    agg[luid]["units"] += h.units
    agg[luid]["cost"] += h.cost.amount if h.cost else 0

print("IBOR-FI aggregated positions:")
for luid, data in agg.items():
    print(f"  {luid}: {data['units']:,.0f} units, cost {data['cost']:,.2f} {data['currency']}")

# Create flat portfolio
try:
    portfolios_api.delete_portfolio(scope=SCOPE, code="IBOR-FI-FLAT")
except: pass

tp_api.create_portfolio(scope=SCOPE,
    create_transaction_portfolio_request=lm.CreateTransactionPortfolioRequest(
        display_name="IBOR FI (Flattened for Recon)", code="IBOR-FI-FLAT",
        base_currency="USD", created="2024-01-01T00:00:00+00:00",
        instrument_scopes=[SCOPE]))

flat_txns = []
for i, (luid, data) in enumerate(agg.items()):
    price = (data['cost'] / data['units'] * 100) if data['units'] > 0 else 0
    flat_txns.append(lm.TransactionRequest(
        transaction_id=f"FLAT-{i+1:03d}", type="Buy",
        instrument_identifiers={"Instrument/default/LusidInstrumentId": luid},
        transaction_date=datetime(2024, 3, 1, tzinfo=timezone.utc).isoformat(),
        settlement_date=datetime(2024, 3, 3, tzinfo=timezone.utc).isoformat(),
        units=data['units'],
        transaction_price=lm.TransactionPrice(price=price, type="Price"),
        total_consideration=lm.CurrencyAndAmount(amount=data['cost'], currency=data['currency']),
        transaction_currency=data['currency']))

tp_api.upsert_transactions(scope=SCOPE, code="IBOR-FI-FLAT", transaction_request=flat_txns)
print(f"\nIBOR-FI-FLAT: {len(flat_txns)} positions with correct costs")

---
## Step 2: Create Custodian Mirror Portfolio
Load custodian reported positions. Note the two deliberate breaks:
* MBS has 4,950,000 units (IBOR has 5,000,000)
* Convertible 2.5% is completely missing

In [ ]:
try:
    portfolios_api.delete_portfolio(scope=SCOPE, code="CUSTODIAN-MIRROR")
except: pass

tp_api.create_portfolio(scope=SCOPE,
    create_transaction_portfolio_request=lm.CreateTransactionPortfolioRequest(
        display_name="Custodian Positions (Mirror)", code="CUSTODIAN-MIRROR",
        base_currency="USD", created="2024-01-01T00:00:00+00:00",
        instrument_scopes=[SCOPE]))

# Load custodian positions (with breaks) as transactions
df_cust = pd.read_csv("data/ibor_custodian_positions.csv")
print(f"Custodian file: {len(df_cust)} positions")
display(df_cust)

cust_txns = []
for i, (_, row) in enumerate(df_cust.iterrows()):
    cust_txns.append(lm.TransactionRequest(
        transaction_id=f"CUST-{i+1:03d}", type="Buy",
        instrument_identifiers={"Instrument/default/ClientInternal": row['ClientInternal']},
        transaction_date=datetime(2024, 3, 1, tzinfo=timezone.utc).isoformat(),
        settlement_date=datetime(2024, 3, 3, tzinfo=timezone.utc).isoformat(),
        units=float(row['Units']),
        transaction_price=lm.TransactionPrice(
            price=float(row['CostAmount']) / float(row['Units']) * 100, type="Price"),
        total_consideration=lm.CurrencyAndAmount(
            amount=float(row['CostAmount']), currency=row['Currency']),
        transaction_currency=row['Currency']))

tp_api.upsert_transactions(scope=SCOPE, code="CUSTODIAN-MIRROR", transaction_request=cust_txns)
print(f"\nLoaded {len(cust_txns)} custodian transactions into mirror")

---
## Step 3: Run Holdings Reconciliation via API
Compares IBOR positions against custodian using the `reconcile_holdings` API.

In [ ]:
result = recon_api.reconcile_holdings(
    portfolios_reconciliation_request=lm.PortfoliosReconciliationRequest(
        left=lm.PortfolioReconciliationRequest(
            portfolio_id=lm.ResourceId(scope=SCOPE, code="IBOR-FI-FLAT"),
            effective_at="2024-09-30"),
        right=lm.PortfolioReconciliationRequest(
            portfolio_id=lm.ResourceId(scope=SCOPE, code="CUSTODIAN-MIRROR"),
            effective_at="2024-09-30"),
        instrument_property_keys=["Instrument/default/Name"]))

print(f"{'Instrument':<40} {'IBOR':>15} {'Custodian':>15} {'Difference':>15} {'Status'}")
print("=" * 105)

matches = breaks = 0
for brk in result.values:
    name = ""
    for prop in brk.instrument_properties:
        if prop.key == "Instrument/default/Name":
            name = prop.value.label_value
    left = brk.left_units
    right = brk.right_units
    diff = brk.difference_units
    if abs(diff) < 0.01:
        status = "MATCH"
        matches += 1
    elif left == 0:
        status = "MISSING IN IBOR"
        breaks += 1
    elif right == 0:
        status = "MISSING IN CUSTODIAN"
        breaks += 1
    else:
        status = "UNIT BREAK"
        breaks += 1
    print(f"{name[:40]:<40} {left:>15,.0f} {right:>15,.0f} {diff:>15,.0f} {status}")

print(f"\nSummary: {matches} matched, {breaks} breaks")

---
## Step 4: Set Up Group Reconciliation (UI Visible)
Create a comparison ruleset and reconciliation definition so results appear in the **Reconciliations > Overview** dashboard in the web app.

In [ ]:
# Create comparison ruleset
try:
    grp_recon_api.create_comparison_ruleset(
        create_group_reconciliation_comparison_ruleset_request=lm.CreateGroupReconciliationComparisonRulesetRequest(
            id=lm.ResourceId(scope=SCOPE, code="IBOR-RECON-RULES"),
            display_name="IBOR Holdings Reconciliation Rules",
            reconciliation_type="Holding",
            core_attribute_rules=[
                lm.GroupReconciliationCoreAttributeRule(
                    left=lm.GroupReconciliationCoreComparisonRuleOperand(
                        key="Instrument/default/Name", operation="Value"),
                    right=lm.GroupReconciliationCoreComparisonRuleOperand(
                        key="Instrument/default/Name", operation="Value"),
                    is_comparison_case_sensitive=False)],
            aggregate_attribute_rules=[
                lm.GroupReconciliationAggregateAttributeRule(
                    left=lm.GroupReconciliationAggregateComparisonRuleOperand(
                        key="Holding/default/Units", operation="Sum"),
                    right=lm.GroupReconciliationAggregateComparisonRuleOperand(
                        key="Holding/default/Units", operation="Sum"),
                    tolerance=lm.GroupReconciliationComparisonRuleTolerance(
                        type="Absolute", value=0.01))]))
    print("Created comparison ruleset: IBOR-RECON-RULES")
except lu.ApiException as e:
    if 'AlreadyExists' in str(e.body):
        print("Ruleset already exists")
    else:
        print(f"Ruleset error: {str(e.body)[:300]}")

# Create reconciliation definition
try:
    grp_recon_api.create_group_reconciliation_definition(
        create_group_reconciliation_definition_request=lm.CreateGroupReconciliationDefinitionRequest(
            id=lm.ResourceId(scope=SCOPE, code="IBOR-FI-RECON"),
            display_name="IBOR FI vs Custodian Reconciliation",
            description="Compares IBOR Fixed Income holdings against custodian reported positions. Deliberate breaks: MBS unit break (50,000) and missing Convertible.",
            portfolio_entity_ids=lm.GroupReconciliationDefinitionPortfolioEntityIds(
                left=[lm.ResourceId(scope=SCOPE, code="IBOR-FI-FLAT")],
                right=[lm.ResourceId(scope=SCOPE, code="CUSTODIAN-MIRROR")]),
            recipe_ids=lm.GroupReconciliationDefinitionRecipeIds(
                left=lm.ResourceId(scope=SCOPE, code="IBOR-VALUATION-RECIPE"),
                right=lm.ResourceId(scope=SCOPE, code="IBOR-VALUATION-RECIPE")),
            currencies=lm.GroupReconciliationDefinitionCurrencies(
                left="USD", right="USD"),
            comparison_ruleset_ids=lm.GroupReconciliationDefinitionComparisonRulesetIds(
                holding_reconciliation=lm.ResourceId(scope=SCOPE, code="IBOR-RECON-RULES"))))
    print("Created definition: IBOR-FI-RECON")
except lu.ApiException as e:
    if 'AlreadyExists' in str(e.body):
        print("Definition already exists")
    else:
        print(f"Definition error: {str(e.body)[:300]}")

# Run reconciliation
try:
    grp_recon_api.run_reconciliation(
        scope=SCOPE, code="IBOR-FI-RECON",
        group_reconciliation_run_request=lm.GroupReconciliationRunRequest(
            instance_id=f"{SCOPE}/IBOR-FI-RECON",
            dates_to_reconcile=lm.GroupReconciliationDates(
                left=lm.GroupReconciliationDatePair(
                    business=datetime(2024, 9, 30, tzinfo=timezone.utc)),
                right=lm.GroupReconciliationDatePair(
                    business=datetime(2024, 9, 30, tzinfo=timezone.utc)))))
    print("Reconciliation run complete!")
except lu.ApiException as e:
    print(f"Run error: {str(e.body)[:300]}")

---
## Step 5: View Results in the Web App

Navigate to **Reconciliations > Valuation** in the LUSID web app:

1. Left: `IBOR-FI-FLAT`, Recipe: `IBOR-VALUATION-RECIPE`, Effective: `30 Sep 2024`
2. Right: `CUSTODIAN-MIRROR`, Recipe: `IBOR-VALUATION-RECIPE`, Effective: `30 Sep 2024`
3. Click **Display diffs** checkbox
4. You should see:
   * **Partially matched**: Most instruments with 0 diff (clean matches)
   * **Unmatched (1)**: Convertible 2.5% March 2027 (missing from custodian)
   * **MBS break**: Diff of ~$5.3M from the 50,000 unit discrepancy

---
## Step 6: Break Analysis

In [ ]:
print("BREAK ANALYSIS")
print("=" * 80)
print()
print("Break 1: Amortising MBS 4% May 2035")
print("  IBOR units:      5,000,000")
print("  Custodian units:  4,950,000")
print("  Difference:       50,000")
print("  Likely cause:     Missed principal paydown of 50,000 face value")
print("  Action:           Check custodian paydown schedule. If confirmed,")
print("                    load a principal paydown transaction in IBOR-FI")
print("                    to reduce the MBS position by 50,000 units.")
print()
print("Break 2: Convertible 2.5% March 2027")
print("  IBOR units:      1,000,000")
print("  Custodian units:  0 (not reported)")
print("  Likely cause:     Custodian failed to report the position, OR")
print("                    the bond was called/converted and IBOR missed the event")
print("  Action:           Contact custodian to confirm position status.")
print("                    If the bond was called, load a BondEarlyRedemption")
print("                    corporate action to remove the position from IBOR.")
print()
print("NB07 Complete. The IBOR Training Pack is finished.")